In [15]:
import pandas as pd
import kagglehub

path = kagglehub.dataset_download(
    "saurabhshahane/ecommerce-text-classification"
)

print(path)

C:\Users\dbere\.cache\kagglehub\datasets\saurabhshahane\ecommerce-text-classification\versions\76


In [16]:
import os

print(os.listdir(path))

['ecommerceDataset.csv']


In [17]:
csv_path = path + "/ecommerceDataset.csv"

df = pd.read_csv(
    csv_path,
    names=["category", "text"],
    header=None,
    encoding="latin1",
    on_bad_lines="skip"
)

print(df.head())
print(df["category"].value_counts())

    category                                               text
0  Household  Paper Plane Design Framed Wall Hanging Motivat...
1  Household  SAF 'Floral' Framed Painting (Wood, 30 inch x ...
2  Household  SAF 'UV Textured Modern Art Print Framed' Pain...
3  Household  SAF Flower Print Framed Painting (Synthetic, 1...
4  Household  Incredible Gifts India Wooden Happy Birthday U...
category
Household                 19313
Books                     11820
Electronics               10621
Clothing & Accessories     8671
Name: count, dtype: int64


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report

df = df.dropna()

X = df["text"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=10000
    )),
    ("svd", TruncatedSVD(
        n_components=300,
        random_state=42
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9419930589985126

Classification Report:
                        precision    recall  f1-score   support

                 Books       0.95      0.93      0.94      2364
Clothing & Accessories       0.96      0.96      0.96      1734
           Electronics       0.95      0.91      0.93      2124
             Household       0.93      0.96      0.94      3863

              accuracy                           0.94     10085
             macro avg       0.95      0.94      0.94     10085
          weighted avg       0.94      0.94      0.94     10085



In [19]:
examples = [
    "Men cotton shirt with casual design",
    "Samsung wireless headphones with bluetooth",
    "Wooden wall shelf for home decoration",
    "Python programming book for beginners"
]

print(model.predict(examples))

['Clothing & Accessories' 'Electronics' 'Household' 'Books']
